# Model Training — Colab GPU workflow

1. Open this notebook at **colab.research.google.com** (upload from your PC if needed).
2. **Runtime → Change runtime type → GPU → A100** (or best available) → Save.
3. **Private repo:** Colab **key icon → Secrets** → add `GITHUB_TOKEN` (fine-grained PAT with **Contents: read** on this repo). Never paste tokens into notebook cells.
4. Run cells top to bottom.

Default job: **Moment Detection** synthetic data (`--count 500`). Story Chunking cells are at the bottom.

In [ ]:
# 1. Verify GPU runtime (must NOT say "command not found")
!nvidia-smi

In [ ]:
# 2. Repo settings — no secrets in this cell
from pathlib import Path

GITHUB_REPO_SLUG = "ClergeF/model-training"
GIT_BRANCH = "main"
REPO_ROOT = Path("/content/model-training")
LLAMA_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"


def github_clone_url() -> str:
    """Use Colab secret GITHUB_TOKEN for private repos; public repos need no token."""
    base = f"https://github.com/{GITHUB_REPO_SLUG}.git"
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN").strip()
        if token:
            return f"https://{token}@github.com/{GITHUB_REPO_SLUG}.git"
    except Exception:
        pass
    return base

In [ ]:
# 3. Clone or update from GitHub (no zip upload)
import os
import shutil

MARKER = REPO_ROOT / "data-generation" / "generate_moment_detection_dataset.py"
CLONE_URL = github_clone_url()

if MARKER.exists():
    os.chdir(REPO_ROOT)
    get_ipython().system("git fetch origin")
    get_ipython().system(f"git checkout {GIT_BRANCH}")
    get_ipython().system(f"git pull origin {GIT_BRANCH}")
    print("Updated existing clone:", REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    get_ipython().system(
        f'git clone --branch {GIT_BRANCH} --depth 1 "{CLONE_URL}" "{REPO_ROOT}"'
    )
    if not MARKER.exists():
        raise FileNotFoundError(
            "Clone failed or repo is empty. Push latest code to GitHub, "
            "or add Colab secret GITHUB_TOKEN for a private repo (see intro cell)."
        )
    print("Cloned:", REPO_ROOT)

os.chdir(REPO_ROOT)
!ls -la data-generation/generate_moment_detection_dataset.py

**Fallback (Google Drive):** mount Drive, copy the project folder under My Drive, then run the optional cell below.

In [ ]:
# OPTIONAL — only if clone fails: find project on Google Drive
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

def find_repo_root() -> Path | None:
    for root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")):
        if not root.exists():
            continue
        for path in root.rglob("generate_moment_detection_dataset.py"):
            if path.parent.name == "data-generation":
                return path.parent.parent
    return None

found = find_repo_root()
if found is None:
    raise FileNotFoundError(
        "No repo on Drive. Copy your project folder into My Drive, or fix GitHub clone above."
    )
REPO_ROOT = found
print("Using Drive copy:", REPO_ROOT)

In [ ]:
# 4. Install dependencies
import importlib.util
import os

os.chdir(REPO_ROOT)

if importlib.util.find_spec("transformers") is None:
    get_ipython().system("pip install -q -r data-generation/requirements.txt")
else:
    print("Dependencies already installed.")

In [ ]:
# 5. Hugging Face — required for Llama (run once per Colab runtime)
# Accept the model license at huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
!huggingface-cli login

In [ ]:
# 6. CUDA + token check
import os

import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA — switch to Colab GPU runtime, not local kernel.")
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

if not os.getenv("HF_TOKEN") and not os.getenv("HUGGING_FACE_HUB_TOKEN"):
    print("WARNING: HF token not in env — run the login cell above if generation fails.")
else:
    print("Hugging Face token: set")

In [ ]:
# 7. Moment Detection — smoke test (3 inputs + labels)
import os

os.chdir(REPO_ROOT)

get_ipython().system(
    f"python data-generation/generate_moment_detection_dataset.py "
    f"--smoke-test --provider transformers --model {LLAMA_MODEL}"
)

In [ ]:
# 8. Moment Detection — overnight run (resume-safe; re-run same cell if disconnected)
import os

os.chdir(REPO_ROOT)

get_ipython().system(
    f"python data-generation/generate_moment_detection_dataset.py "
    f"--count 500 --phase all --provider transformers --model {LLAMA_MODEL} "
    f"--inputs moment-detection/data/training/synthetic_moment_inputs.jsonl "
    f"--outputs moment-detection/data/training/synthetic_moment_outputs.jsonl"
)

In [ ]:
# 9. Copy results to Google Drive before runtime dies
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

export_dir = Path("/content/drive/MyDrive/model-training-exports/moment-detection")
export_dir.mkdir(parents=True, exist_ok=True)
training = REPO_ROOT / "moment-detection" / "data" / "training"
for name in ("synthetic_moment_inputs.jsonl", "synthetic_moment_outputs.jsonl"):
    src = training / name
    if src.exists():
        shutil.copy2(src, export_dir / name)
        print("Copied:", export_dir / name)
    else:
        print("Missing (not generated yet):", src)

---
## Optional: Story Chunking V2

In [ ]:
# Story Chunking V2 — smoke (1 example)
import os

os.chdir(REPO_ROOT)

get_ipython().system(
    f"python data-generation/generate_story_chunking_dataset.py --smoke-test "
    f"--provider transformers --model {LLAMA_MODEL} "
    f"--output story-chunking/data/training/v2/synthetic_story_chunking_v2.jsonl"
)

In [ ]:
# Story Chunking V2 — full 20-example run
import os

os.chdir(REPO_ROOT)

get_ipython().system(
    f"python data-generation/generate_story_chunking_dataset.py --count 20 "
    f"--provider transformers --model {LLAMA_MODEL} "
    f"--output story-chunking/data/training/v2/synthetic_story_chunking_v2.jsonl"
)